In [2]:
import os
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.special import softmax
import torch
from tqdm.auto import tqdm
# Register tqdm with pandas
tqdm.pandas()

In [3]:
# 2. Setup Model and GPU
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [4]:
def get_multilingual_sentiment(text):
    """Returns a score between -1 (Negative) and 1 (Positive)"""
    if not text or len(str(text)) < 5:
        return 0
    
    try:
        # Preprocess and tokenize
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        
        # Get model output
        with torch.no_grad():
            output = model(**encoded_input)
        
        # Convert output to probabilities
        scores = output[0][0].detach().cpu().numpy()
        scores = softmax(scores)
        
        # XLM-T output is: [Negative, Neutral, Positive]
        # Calculate a weighted compound score
        ranking = scores[2] - scores[0] # (Positive - Negative)
        return ranking
    except:
        return 0




def process_file(file_path, output_name):
    # 1. FIX: Create the folder BEFORE the long loop starts
    target_folder = "translated_data"
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"Created directory: {target_folder}")
    
    df = pd.read_csv(file_path, lineterminator='\n')
        
    print(f"Starting sentiment analysis for {output_name}...")

    # .progress_apply() creates a live progress bar in Colab
    df['sentiment_score'] = df['tweet'].progress_apply(get_multilingual_sentiment)
    
    df.to_csv(os.path.join(target_folder, output_name), index=False)
    print(f"saved to {output_name}")

process_file('/content/trump_preprocessed.csv', 'trump_translated.csv')
process_file('/content/biden_preprocessed.csv', 'biden_translated.csv')


Starting sentiment analysis for trump_translated.csv...


  0%|          | 0/320620 [00:00<?, ?it/s]

OSError: Cannot save file into a non-existent directory: 'processed_data'

In [ ]:
import os

# Create the missing directory
os.makedirs('processed_data', exist_ok=True)

# Save the dataframe that is currently in memory
df.to_csv('./processed_data/trump_translated.csv', index=False)
print("Phew! Trump data saved successfully to ./processed_data/trump_translated.csv")